## **LD50 database generation**

In [1]:
import os
from pathlib import Path
import sys

# Define xenotox as base directory
BASE_DIR = Path(f"{os.getcwd()}/..").resolve()

# Add parent directory to sys.path for imports
sys.path.append(str(BASE_DIR))

In [ ]:
from utils_reg.ld50_curation import combine_csv
# Define path
DATA_DIR = f"{BASE_DIR}/ld50_data"
columns = ["Molecule name", "SMILES", "LD50", "DB source",
            "Risk assessment class", "Species", "Exposure route", "Standard value"]
# Combine CSV files
df = combine_csv(DATA_DIR, columns)
display(df.head())

,Molecule name,SMILES,LD50,DB source,Risk assessment class,Species,Exposure route,Standard value
0,Formaldehyde,C=O,260.0,ChemIDplus,acute,rat,oral,=
1,Formaldehyde,C=O,42.0,ChemIDplus,acute,rat,oral,=
2,Formaldehyde,C=O,232.47,ChemIDplus,acute,ert,dermal,=
3,Formaldehyde,C=O,100.0,ChemIDplus,acute,rat,oral,=
4,Chlorpromazine,NaN,135.0,ChemIDplus,acute,rat,oral,=


In [ ]:
from utils_reg.ld50_curation import curate_data

df_curated, curation_report, standardization_report, failed_smiles = curate_data(df, smiles_col="SMILES", target_col="LD50")
display(df_curated.head())
display(curation_report)
display(standardization_report)
display(failed_smiles)

,Molecule name,SMILES,LD50,DB source,Risk assessment class,Species,Exposure route,Standard value,SMILES_raw
0,"2,3,7,8-Tetrachlorodibenzo-p-dioxin",Clc1cc2c(cc1Cl)Oc1cc(Cl)c(Cl)cc1O2,0.0005,ChemIDplus,acute,rat,oral,=,Clc(c(Cl)c1)cc(Oc2c3)c1Oc2cc(Cl)c3Cl
1,"Phosphorothioic acid, O,O-diethyl S-((5-methyl...",CCOP(=O)(OCC)SCc1nnc(C)o1,0.0030,TEST,acute,rat,oral,=,CCOP(OCC)(SCc1nnc(C)[o]1)=O
2,"1,2,3,7,8-Pentachlorodibenzo-p-dioxin",Clc1cc2c(cc1Cl)Oc1c(cc(Cl)c(Cl)c1Cl)O2,0.0031,ChemIDplus,acute,rat,oral,=,Clc(c(Cl)c1)cc2c1Oc(c(Cl)c(c(Cl)c1)Cl)c1O2
3,"O,O-Diethyl S-[(5-methyl-1,3,4-oxadiazol-2-yl)...",CCOP(=S)(OCC)SCc1nnc(C)o1,0.0050,TEST,acute,rat,oral,=,CCOP(OCC)(SCc1nnc(C)[o]1)=S
4,"2,3,7,8-Tetrachlorodibenzofuran",Clc1cc2oc3cc(Cl)c(Cl)cc3c2cc1Cl,0.0050,ChemIDplus,acute,rat,oral,=,Clc(c(Cl)c1)cc(c2c3)c1[o]c2cc(Cl)c3Cl


,Step,Description,Entries_before,Entries_after,Removed,Removed_%
0,1.0,"Semantic filtering (acute, rat, oral, exact va...",129416,60827,68589,53.00
1,2.0,Target cleaning (numeric),60827,60727,100,0.16
2,3.0,SMILES parsing,60727,40551,20176,33.22
3,3.5,Sanitization filter,40551,40551,0,0.00
4,4.0,Filtering non-organic molecules,40551,38499,2052,5.06
5,5.0,Structural standardization (failed removed),38499,38493,6,0.02
6,6.0,Canonical SMILES generation,38493,38488,5,0.01
7,7.0,Deduplication (lowest LD50 kept),38488,19238,19250,50.02


,Substep,Molecules,Changed,Changed_%
0,Normalize,38499,134,0.35
1,FragmentParent,38499,6695,17.39
2,Uncharger,38499,2338,6.07
3,TautomerCanonical,38499,3603,9.36


5912              CCCCCCCCCCCCCCP(=O)=[OH+]
8213                          COP(=O)=[OH+]
8379     NC(Cc1c([O-])[o]n[n+]1Cc1ccccc1)=O
36655                         COP(=O)=[OH+]
36977    NC(Cc1c([O-])[o]n[n+]1Cc1ccccc1)=O
38145          Cc1[n+](-c2ccccc2)[o]nc1[O-]
Name: SMILES_raw, dtype: object

In [ ]:
# Save curated data
df_curated.to_csv(f"{BASE_DIR}/ld50_data/ld50_db.csv", index=False)
curation_report.to_csv(f"{BASE_DIR}/ld50_data/curation_report.csv", index=False)
standardization_report.to_csv(f"{BASE_DIR}/ld50_data/standardization_report.csv", index=False)